# QLoRA Fine-Tuning — Qwen3 1.7B (Mental-Health Counselor)

**Objective:** QLoRA fine-tune the **instruct** model Qwen3 1.7B on BOTH study datasets (3 models × 2 datasets study). Run top-to-bottom: each dataset is trained **separately** — a fresh 4-bit model, its own trainer, its own adapter — one after the other, with GPU memory freed in between.

| Item | Detail |
|------|--------|
| **Model** | `Qwen/Qwen3-1.7B` — instruct model, QLoRA (the study design) |
| **Run 1** | `amod` — Amod counseling (~3.5K) → `../models/chat_model/amod/qwen3/` |
| **Run 2** | `mentalchat16k` — MentalChat16K (~16K) → `../models/chat_model/mentalchat16k/qwen3/` |
| **Recipe** | QLoRA 4-bit NF4 + double quant + bf16 · LoRA r=16, α=32, dropout 0.05 on q/k/v/o/gate/up/down_proj · 3 epochs · eff. batch 8 (2 × 4 grad-accum) · cosine LR 2e-4 · packing · max_len from config |
| **Safety** | curated crisis examples ×3 injected into the train split only |
| **Access** | Apache-2.0, ungated — good first run |
| **Note** | `/no_think` soft switch keeps Qwen3 in non-thinking mode (same approach as the prior study). |

All hyperparameters resolve from `config.yaml` via `ft_utils`; each run saves adapter + tokenizer + `run_info.json`.

In [1]:
# Cell 1 — Imports
import os
os.environ["PYTHONUTF8"] = "1"  # Windows charmap fix for TRL
os.environ["BNB_CUDA_VERSION"] = "130"  # torch is cu132; bitsandbytes 0.49.2 has binaries only up to cu130

import gc
import json
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model
from trl import SFTTrainer, SFTConfig

import ft_utils

In [2]:
# Cell 2 — Model setup shared by both runs (config, tokenizer, LoRA/quant)
MODEL_KEY = "qwen3"

MCFG = ft_utils.CFG["models"][MODEL_KEY]
TCFG = ft_utils.CFG["training"]
MODEL_ID = MCFG["instruct_id"]   # QLoRA on the INSTRUCT model

QCFG = TCFG["quant"]
bnb_config = BitsAndBytesConfig(
    load_in_4bit=QCFG["load_in_4bit"],
    bnb_4bit_quant_type=QCFG["bnb_4bit_quant_type"],
    bnb_4bit_compute_dtype=getattr(torch, QCFG["bnb_4bit_compute_dtype"]),
    bnb_4bit_use_double_quant=QCFG["bnb_4bit_use_double_quant"],
)

LCFG = TCFG["lora"]
lora_config = LoraConfig(
    r=LCFG["r"],
    lora_alpha=LCFG["alpha"],
    lora_dropout=LCFG["dropout"],
    bias="none",
    target_modules=list(LCFG["target_modules"]),
    task_type=TaskType.CAUSAL_LM,
)

model_kwargs = dict(
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,
)
if MCFG.get("attn_implementation"):
    model_kwargs["attn_implementation"] = MCFG["attn_implementation"]  # Gemma 2: eager

sft_kwargs = {k: v for k, v in TCFG.items() if k not in ("lora", "quant", "max_seq_length")}

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model:  {MCFG['family']} ({MODEL_ID})")
print(f"Method: QLoRA | LoRA r={LCFG['r']} alpha={LCFG['alpha']} | max_len {TCFG['max_seq_length']}")

Model:  Qwen3 1.7B (Qwen/Qwen3-1.7B)
Method: QLoRA | LoRA r=16 alpha=32 | max_len 1024


In [3]:
# Cell 3 — One full training run: data -> fresh 4-bit model -> LoRA -> train
#          -> save adapter -> smoke test -> free VRAM
def train_one(dataset_key):
    run_name = f"{MODEL_KEY}_{dataset_key}"
    adapter_out = ft_utils.MODELS_ROOT / dataset_key / MODEL_KEY  # chat_model/<dataset>/<model>/
    ckpt_dir = ft_utils.ROOT / "checkpoints" / run_name
    print("=" * 70)
    print(f"RUN {run_name} — {ft_utils.CFG['datasets'][dataset_key]['name']}")
    print("=" * 70)

    # Data (crisis examples injected into train per config)
    train_ds, val_ds, test_ds_raw = ft_utils.get_formatted(MODEL_KEY, dataset_key)
    print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test (raw): {len(test_ds_raw)}")
    print(f"Sample formatted text:\n{train_ds[0]['text'][:400]}\n")

    # Fresh quantized model + LoRA for this dataset
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)
    model.config.use_cache = False
    model = get_peft_model(model, lora_config)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

    sft_config = SFTConfig(
        output_dir=str(ckpt_dir),
        max_length=TCFG["max_seq_length"],
        dataset_text_field="text",
        gradient_checkpointing_kwargs={"use_reentrant": False},
        **sft_kwargs,
    )
    trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        args=sft_config,
    )

    start = time.time()
    trainer.train()
    minutes = (time.time() - start) / 60
    print(f"\nTraining complete in {minutes:.1f} minutes")

    # Save adapter + tokenizer + manifest
    adapter_out.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(adapter_out)
    tokenizer.save_pretrained(adapter_out)
    manifest = {
        "run_name": run_name,
        "model_key": MODEL_KEY,
        "family": MCFG["family"],
        "dataset_key": dataset_key,
        "method": "qlora",
        "model_id": MODEL_ID,
        "seed": ft_utils.SEED,
        "max_seq_length": TCFG["max_seq_length"],
        "train_minutes": round(minutes, 1),
        "final_train_loss": next((h["train_loss"] for h in reversed(trainer.state.log_history)
                                  if "train_loss" in h), None),
        "best_eval_loss": min((h["eval_loss"] for h in trainer.state.log_history
                               if "eval_loss" in h), default=None),
    }
    with open(adapter_out / "run_info.json", "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
    print(f"Saved adapter + run_info.json to {adapter_out}")

    # Smoke test on 2 held-out prompts
    model.eval()
    model.config.use_cache = True
    format_prompt = ft_utils.PROMPT_FORMATTERS[MODEL_KEY]
    for idx in range(2):
        user_input = test_ds_raw[idx]["input"]
        # add_special_tokens=False: the template already carries BOS (matches TRL training)
        inputs = tokenizer(format_prompt(user_input), return_tensors="pt",
                           add_special_tokens=False).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
        reply = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        print(f"\n--- Sample {idx + 1} ---")
        print(f"USER:  {user_input[:200]}")
        print(f"MODEL: {reply[:400]}")

    # Free VRAM for the next run
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()
    return manifest

In [4]:
# Cell 4 — Run 1: amod (Amod counseling)
info_A = train_one("amod")

RUN qwen3_amod — Mental Health Counseling Conversations (Amod)


Map:   0%|          | 0/1763 [00:00<?, ? examples/s]

Map:   0%|          | 0/496 [00:00<?, ? examples/s]

Train: 1763 | Val: 496 | Test (raw): 244
Sample formatted text:
<|im_start|>system
/no_think
You are a supportive, empathetic mental health counseling assistant. Respond with warmth and care. If the user expresses thoughts of self-harm or suicide, gently encourage them to reach out to a crisis line or mental health professional rather than attempting to resolve it yourself.<|im_end|>
<|im_start|>user
I am a survivor of domestic violence from a past relationshi



This can be used to load a bitsandbytes version built with a CUDA version that is different from the PyTorch CUDA version.
If this was unintended set the BNB_CUDA_VERSION variable to an empty string: export BNB_CUDA_VERSION=

W0722 22:14:55.837000 19864 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


...

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,1.997804,2.032606,1.892842,635807.000000,0.550099
2,1.749903,1.984592,1.854186,1271614.000000,0.562023
3,1.571613,2.022308,1.721360,1907421.000000,0.561217



Training complete in 42.7 minutes
Saved adapter + run_info.json to C:\Users\laoli\OneDrive\Desktop\MultiAgent2\models\chat_models\amod\qwen3


c:\Users\laoli\OneDrive\Desktop\MultiAgent2\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



--- Sample 1 ---
USER:  Is it normal for people to cry during therapy, or is it just me?
MODEL: It is normal to cry during therapy.  It is a sign that you are feeling something.  It is also a sign that you are open to therapy.  It is important to remember that you are not alone in this.  Many people cry during therapy.  It is also important to remember that crying is not a sign of weakness.  It is a sign that you are feeling something.  It is also important to remember that crying is not a s

--- Sample 2 ---
USER:  Is it normal for people to cry during therapy, or is it just me?
MODEL: It is normal to cry during therapy.  It is a sign that you are feeling something.  It is also a sign that you are open to therapy.  It is important to remember that you are not alone in this.  Many people cry during therapy.  It is also important to remember that crying is not a sign of weakness.  It is a sign that you are feeling something.  It is also important to remember that crying is not a s


In [5]:
# Cell 5 — Run 2: mentalchat16k (MentalChat16K)
info_B = train_one("mentalchat16k")

RUN qwen3_mentalchat16k — MentalChat16K (ShenLab)


Map:   0%|          | 0/11221 [00:00<?, ? examples/s]

Map:   0%|          | 0/3196 [00:00<?, ? examples/s]

Train: 11221 | Val: 3196 | Test (raw): 1597
Sample formatted text:
<|im_start|>system
/no_think
You are a supportive, empathetic mental health counseling assistant. Respond with warmth and care. If the user expresses thoughts of self-harm or suicide, gently encourage them to reach out to a crisis line or mental health professional rather than attempting to resolve it yourself.<|im_end|>
<|im_start|>user
Lately, I have been feeling extremely stressed and overwhelm



Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

...

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.884616,0.870775,0.875562,6146588.000000,0.740748
2,0.814539,0.840613,0.803192,12293176.000000,0.748606
3,0.762960,0.842835,0.758881,18439764.000000,0.748882



Training complete in 317.5 minutes
Saved adapter + run_info.json to C:\Users\laoli\OneDrive\Desktop\MultiAgent2\models\chat_models\mentalchat16k\qwen3


c:\Users\laoli\OneDrive\Desktop\MultiAgent2\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



--- Sample 1 ---
USER:  Patient's message about Parenting.
"Being a new parent has been incredibly overwhelming, and I hope to gain some guidance through counseling. I've been experiencing a mix of joy and anxiety since my b
MODEL: Parenting can indeed be overwhelming, and it's great that you're seeking guidance through counseling. It sounds like you're experiencing a mix of emotions, including joy and anxiety, since your baby was born. The fact that negative emotions seem to dominate lately is understandable given the challenges you're facing.

One approach that may help you manage these feelings better is practicing self-c

--- Sample 2 ---
USER:  As a parent, I often feel overwhelmed and unsure if I'm doing the right thing. My child is going through a rebellious phase, and it's causing a lot of tension in our household. I love them dearly, but
MODEL: Parenting can definitely be overwhelming at times, and it's completely normal to feel unsure about whether you're doing the right thi

In [6]:
# Cell 6 — Recap of both runs
for info in (info_A, info_B):
    print(f"{info['run_name']}: {info['train_minutes']} min | "
          f"final train loss {info['final_train_loss']} | "
          f"best eval loss {info['best_eval_loss']}")

qwen3_amod: 42.7 min | final train loss 1.839948478687223 | best eval loss 1.9845921993255615
qwen3_mentalchat16k: 317.5 min | final train loss 0.8463166000781305 | best eval loss 0.8406127095222473


## Training summary — Qwen3 1.7B

| Item | Value |
|------|-------|
| Runs | `qwen3_amod` and `qwen3_mentalchat16k`, trained separately (fresh model each) |
| Recipe | QLoRA 4-bit NF4 · LoRA r=16 α=32 dropout 0.05 · 3 epochs · eff. batch 8 · cosine 2e-4 · packing · max_len from config |
| Safety | curated crisis examples ×3 injected into the train split only |
| Times / losses | *(see Cell 6 recap; per-run manifests in each adapter's `run_info.json`)* |
| Adapters | `../models/chat_model/amod/qwen3/` · `../models/chat_model/mentalchat16k/qwen3/` |

Evaluation (test-split metrics + safety-eval prompts vs the un-tuned baseline) comes in a later notebook.